## Lec 4. 现代卷积神经网络

- LeNet `卷积神经网络` (1990.)
- AlexNet `深度卷积神经网络`  (2012.)
- VGG `模块化的卷积神经网络` (2014.)
- ResNet `残差结构的卷积神经网络` (2016.)
- DenseNet `稠密卷积神经网络` (2017.)
- FAST-CNN / FAST-RCNN .....
- YOLOv1 - YOLOv12 (2016. - 2025.)

### 4.1 AlexNet

- `ImageNet 数据集` (3 x 224 x 224)
- 1个11x11的大卷积层，然后跟着5x5中型卷积层，再紧接着连续三个3x3的卷积层
- 激活函数从 Sigmoid 换成了 ReLU

![AlexNet网络结构图](https://zh.d2l.ai/_images/alexnet.svg)


In [1]:
import torch
import torch.nn as nn

In [45]:
class AlexNet(nn.Module):
    """ AlexNet 深层卷积神经网络 """
    def __init__(self):
        super().__init__()
        # 卷积层
        self.conv = nn.Sequential(nn.Conv2d(1, 96, kernel_size=11, stride=4, padding=1), 
                                  nn.ReLU(), 
                                  nn.MaxPool2d(kernel_size=3, stride=2), 
                                  
                                  nn.Conv2d(96, 256, kernel_size=5, padding=2), 
                                  nn.ReLU(), 
                                  nn.MaxPool2d(kernel_size=3, stride=2), 
                                  
                                  # 使用3个连续的 3x3 卷积层
                                  nn.Conv2d(256, 384, kernel_size=3, padding=1), 
                                  nn.ReLU(), 
                                  nn.Conv2d(384, 384, kernel_size=3, padding=1), 
                                  nn.ReLU(), 
                                  nn.Conv2d(384, 256, kernel_size=3, padding=1), 
                                  nn.ReLU(), 
                                  nn.MaxPool2d(kernel_size=3, stride=2), 
                                  
                                  nn.Flatten())
        
        self.fc = nn.Sequential(nn.Linear(5*5*256, 4096), 
                                nn.ReLU(), 
                                nn.Linear(4096, 4096), 
                                nn.ReLU(), 
                                nn.Linear(4096, 10))
        
    def forward(self, x):
        """ 前向传播方法 """
        x_flatten = self.conv(x)
        return self.fc(x_flatten)

In [47]:
x = torch.rand(16, 1, 224, 224)
output = model(x)
output.shape

torch.Size([16, 10])

In [77]:
y_true += (torch.tensor([9, 0, 1,2,3,4,5,6,7,8,9,0, 1,2,3,4]) == torch.argmax(output, dim=-1)).sum().item()

2

#### 4.1.1 DataLoader

In [85]:
import torchvision
import torchvision.transforms as transforms
from torch.utils import data
from torch.utils.data import DataLoader

trans = transforms.Compose([transforms.ToTensor(), 
                            transforms.Resize((224, 224))])

mnist_train = torchvision.datasets.FashionMNIST(root="../data", train=True, transform=trans, download = True)
mnist_test = torchvision.datasets.FashionMNIST(root="../data", train=False, transform=trans, download = True)

len(mnist_train), len(mnist_test)

(60000, 10000)

In [91]:
train_dataloader = DataLoader(mnist_train, batch_size=256, shuffle=True)
test_dataloader = DataLoader(mnist_test, batch_size=256, shuffle=False)

len(train_dataloader), len(test_dataloader)

(235, 40)

#### 4.1.2 Model

In [54]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
device

device(type='cpu')

In [55]:
model = AlexNet().to(device)
model

AlexNet(
  (conv): Sequential(
    (0): Conv2d(1, 96, kernel_size=(11, 11), stride=(4, 4), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(96, 256, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(256, 384, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): Conv2d(384, 384, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): ReLU()
    (10): Conv2d(384, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU()
    (12): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
    (13): Flatten(start_dim=1, end_dim=-1)
  )
  (fc): Sequential(
    (0): Linear(in_features=6400, out_features=4096, bias=True)
    (1): ReLU()
    (2): Linear(in_features=4096, out_features=4096, bias=True)
    (3): ReLU()
    (4): Linear(in_featu

In [102]:
def count_parameters(model: nn.Module):
    """ 计算模型参数量函数 """
    return sum(p.numel() for p in model.parameters())

In [103]:
count_parameters(model)

46764746

#### 4.1.3 Train & Test Model

In [87]:
x, y = next(iter(train_dataloader))
x.shape, y.shape

(torch.Size([256, 1, 224, 224]), torch.Size([256]))

In [94]:
def train_model(model, train_dataloader, loss_func, optimizer):
    """ 模型训练函数 """
    model.train()
    total_loss = 0.
    for x, y in train_dataloader:
        # x: [bs, 1, 224, 224]
        # y: [batch_size]
        y_hat = model(x.to(device))
        loss = loss_func(y_hat, y.to(device))
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    return total_loss / len(train_dataloader)

def test_model(model, test_dataloader, loss_func):
    """ 模型测试函数 """
    model.eval()
    
    y_true = 0
    total_loss = 0.
    for x, y in test_dataloader:
        # x: [bs, 1, 224, 224]
        # y: [batch_size]
        y_hat = model(x.to(device))
        loss = loss_func(y_hat, y.to(device))
        
        y_true += (y == torch.argmax(y_hat, dim=-1)).sum().item()
        
        total_loss += loss.item()

    avg_loss = total_loss / len(train_dataloader)
    acc = round(y_true / len(test_dataloader.dataset), 3)
    return avg_loss, acc

In [95]:
model = AlexNet().to(device)
loss_func = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [79]:
n_epoch = 5

train_loss_list = []
test_loss_list = []
for i in range(n_epoch):
    train_loss = train_model(model, train_dataloader, loss_func, optimizer)
    test_loss, acc = test_model(model, train_dataloader, loss_func)
    
    train_loss_list.append(train_loss)
    test_loss_list.append(test_loss)
    print(train_loss)

KeyboardInterrupt: 

In [ ]:
import matplotlib.pyplot as plt
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'

plt.figure(figsize=(12, 8))
plt.plot(train_loss_list, label="train loss")
plt.plot(test_loss_list, label="test loss")
plt.title("Model Loss")
plt.grid()
plt.legend()
plt.show()

### 4.2 VGG 视觉几何组（visual geometry group）

- 模块化的卷积神经网络

![VGG网络结构图](https://zh.d2l.ai/_images/vgg.svg)



In [117]:
def vgg_block(num_conv: int, in_channels: int, out_channels: int) -> nn.Module:
    """ VGG块构建函数 """
    layers = []
    for i in range(num_conv):
        layers.append(nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1))
        layers.append(nn.ReLU())
        in_channels = out_channels
    layers.append(nn.MaxPool2d(kernel_size=2, stride=2))
    return nn.Sequential(*layers)

In [131]:
def vgg(conv_arch: list) -> nn.Module:
    """ VGG网络构建函数 """
    conv_block = []
    in_channels = 1
    # 卷积部分
    for (num_conv, out_channels) in conv_arch:
        conv_block.append(vgg_block(num_conv, in_channels, out_channels))
        in_channels = out_channels
    
    # 全连接部分
    model = nn.Sequential(*conv_block, 
                          nn.Flatten(), 
                          
                          nn.Linear(7*7*out_channels, 4096), 
                          nn.ReLU(), 
                          nn.Linear(4096, 4096), 
                          nn.ReLU(), 
                          nn.Linear(4096, 10))
    return model

In [136]:
conv_arch = [(1, 64), (1, 128), (2, 256), (2, 512), (2, 512)]
model = vgg(conv_arch)
model

Sequential(
  (0): Sequential(
    (0): Conv2d(1, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (1): Sequential(
    (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (2): Sequential(
    (0): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU()
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (3): Sequential(
    (0): Conv2d(256, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): Conv2d(512, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU()
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (4):

In [137]:
x = torch.rand(16, 1, 224, 224)
output = model(x)
output.shape

torch.Size([16, 10])

In [138]:
count_parameters(model)

128806154

### 4.3 ResNet (Residual Network)

![ResNet](https://zh.d2l.ai/_images/residual-block.svg)

0. Github - DNN101_Course / PYH101
1. LeNet (Fasion-MNIST / ImageNet)
2. AlexNet (Fasion-MNIST / ImageNet)
3. VGG (Fasion-MNIST / ImageNet)